Priyanshu Rathor & ______Trevor Henderson______& ____Gabriel Machorro_____________



https://www.kaggle.com/datasets/talhaanjum0/coffee-shop-revenue

In [ ]:
#add libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics

from google.colab import files
uploaded = files.upload()



Saving coffee_shop_revenue.csv to coffee_shop_revenue (3).csv


In [ ]:
# load dataset
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
df.head()

,Number_of_Customers_Per_Day,Average_Order_Value,Operating_Hours_Per_Day,Number_of_Employees,Marketing_Spend_Per_Day,Location_Foot_Traffic,Daily_Revenue
0,152,6.74,14,4,106.62,97,1547.81
1,485,4.50,12,8,57.83,744,2084.68
2,398,9.09,6,6,91.76,636,3118.39
3,320,8.48,17,4,462.63,770,2912.20
4,156,7.44,17,2,412.52,232,1663.42


In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df['Revenue_Category'] = pd.cut(
    df['Daily_Revenue'],
    bins=[-np.inf, 500, 1000, np.inf],
    labels=[0,1,2]
)


In [ ]:
df = df.dropna(subset=['Revenue_Category'])

In [ ]:
df['Revenue_Category'] = df['Revenue_Category'].astype(int)

In [ ]:
#these inputs will be used to predict category
x_data = df.drop(['Daily_Revenue','Revenue_Category'], axis=1)
y_data = df['Revenue_Category']

In [ ]:
#to convert tect columns into numbers
for col in x_data.select_dtypes(include=['object']).columns:
    x_data[col] = x_data[col].astype('category').cat.codes


In [ ]:
#to split into training and testing
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3, random_state=1)

In [ ]:
#train decision tree through creating classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [ ]:
#to calculates how accurate the decision tree is
dt_accuracy = dt_model.score(X_test, y_test)
print("Decision Tree Test Accuracy:", f"{dt_accuracy * 100:.2f}%")

Decision Tree Test Accuracy: 85.17%


In [ ]:
#train random forest using training data
rf_model = RandomForestClassifier(n_estimators=5, max_features=2, random_state=42)

rf_model.fit(X_train, y_train)

RandomForestClassifier(max_features=2, n_estimators=5, random_state=42)

In [ ]:
#to calculates random forest accuracy on the test data
rf_accuracy = rf_model.score(X_test, y_test)
print("RF Model Accuracy:", f"{rf_accuracy * 100:.2f}", "%")

RF Model Accuracy: 83.00 %


In [ ]:
#we again repeat
rf_model = RandomForestClassifier(n_estimators=5, max_features=2, random_state=42)

rf_model.fit(X_train, y_train)

RandomForestClassifier(max_features=2, n_estimators=5, random_state=42)

In [ ]:
#to predicts revenue category for the test data using random forest
y_pred_rf = rf_model.predict(X_test)

In [ ]:
#Calculate precision, recall, and F1-score for Random Forest
precision_rf = metrics.precision_score(y_test, y_pred_rf, average='weighted')
recall_rf = metrics.recall_score(y_test, y_pred_rf, average='weighted')
f1_rf = metrics.f1_score(y_test, y_pred_rf, average='weighted')

print("RF Precision:", precision_rf)
print("RF Recall:", recall_rf)
print("RF F1-Score:", f1_rf)

RF Precision: 0.8179277835527836
RF Recall: 0.83
RF F1-Score: 0.8222416322542972


In [ ]:
#we use this to find average revenue
average_revenue = df.groupby('Revenue_Category')['Daily_Revenue'].mean()

print("Average Revenue per Category:")
print(average_revenue)

labels = {0: "Low Revenue", 1: "Medium Revenue", 2: "High Revenue"}

for category, value in average_revenue.items():
    print(f"{labels[category]} average revenue: {value:.2f}")

Average Revenue per Category:
Revenue_Category
0     366.567818
1     798.646372
2    2206.568823
Name: Daily_Revenue, dtype: float64
Low Revenue average revenue: 366.57
Medium Revenue average revenue: 798.65
High Revenue average revenue: 2206.57


In [ ]:
#predict function the show feature name and then collect user input, predict revenue and print. Repeat for another response
def predict_revenue():
    while True:
        try:
            print("Enter values in this same order:")
            print(list(x_data.columns))

            user_values = []
            for col in x_data.columns:
                value = float(input(f"Enter {col}: "))
                user_values.append(value)

            new_shop_data = pd.DataFrame([user_values], columns=x_data.columns)

            prediction = rf_model.predict(new_shop_data)[0]

            predicted_category_label = labels[prediction]
            estimated_revenue = average_revenue.get(prediction, 'N/A')

            print(f"Prediction: {predicted_category_label}")
            if estimated_revenue != 'N/A':
                print(f"Estimated Revenue for this category: {estimated_revenue:.2f}")
            else:
                print("Could not estimate revenue for this category.")

            another_shop = input("Enter data for another shop? (yes/no): ")
            if another_shop.lower() != 'yes':
                break

        except ValueError:
            print("Invalid input. Please enter valid numerical values.")

predict_revenue()

Enter values in this same order:
['Number_of_Customers_Per_Day', 'Average_Order_Value', 'Operating_Hours_Per_Day', 'Number_of_Employees', 'Marketing_Spend_Per_Day', 'Location_Foot_Traffic']
Enter Number_of_Customers_Per_Day: 0
Enter Average_Order_Value: 0
Enter Operating_Hours_Per_Day: 0
Enter Number_of_Employees: 0
Enter Marketing_Spend_Per_Day: 0
Enter Location_Foot_Traffic: 0
Prediction: Medium Revenue
Estimated Revenue for this category: 798.65
Enter data for another shop? (yes/no): yes 


In [ ]:
feature_importances = pd.DataFrame({'feature': x_data.columns, 'importance': rf_model.feature_importances_})
feature_importances = feature_importances.sort_values('importance', ascending=False)
feature_importances

,feature,importance
0,Number_of_Customers_Per_Day,0.418255
1,Average_Order_Value,0.214212
4,Marketing_Spend_Per_Day,0.163670
5,Location_Foot_Traffic,0.090019
3,Number_of_Employees,0.062425
2,Operating_Hours_Per_Day,0.051419


We used the coffee shop dataset to classify shops into three revenue group are low revenue, medium revenue, and high revenue .Firstly, we loaded the dataset to cleaned the column names, and created a new target column called revenue category based on Daily revenue values and then we separated the input features and target, converted text columns into numbers and split the data into training and testing parts. After that we trained two machine learning models which is decision tree and random forest and compared their accuracy. We also measured precision, recall and F1 score to see how well the Random Forest model performed and then we made the program interactive so the user can enter new shop information and get a predicted revenue category along with an estimated revenue value. We created a feature importance table to show which shop features have the biggest impact on revenue. This is helpful in real life because it can help business owners understand whether a coffee shop is likely to make low, medium, or high revenue and which factors matter most for improving business performance.